In [1]:
!pip3 install hmmlearn

In [2]:
import pandas as pd
from hmmlearn import hmm
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report

In [3]:
df = pd.read_csv('dataset/events.csv', sep=',')
# Sort entries by visitorid and time
df = df.sort_values(by=['visitorid', 'timestamp'])
# Change time to readable date & time 
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
df.head()

,timestamp,visitorid,event,itemid,transactionid
1361687,2015-09-11 20:49:49.439,0,view,285930,NaN
1367212,2015-09-11 20:52:39.591,0,view,357564,NaN
1367342,2015-09-11 20:55:17.175,0,view,67045,NaN
830385,2015-08-13 17:46:06.444,1,view,72028,NaN
742616,2015-08-07 17:51:44.567,2,view,325215,NaN


## Hyperparameter Tuning: Session Threshold
We compare different session timeout thresholds (30, 60, and 120 minutes) to determine the optimal value for our model. A session is defined as a sequence of events by the same visitor with no gap larger than the threshold.

In [4]:
def run_experiment(threshold_minutes, df_input):
    print(f"--- Running Experiment with Threshold: {threshold_minutes} minutes ---")
    df_exp = df_input.copy()
    
    # Sessionization
    df_exp['time_diff'] = df_exp.groupby('visitorid')['timestamp'].diff()
    session_threshold = pd.Timedelta(minutes=threshold_minutes)
    df_exp['new_session'] = (df_exp['time_diff'].isna()) | (df_exp['time_diff'] > session_threshold)
    df_exp['session_id'] = df_exp.groupby('visitorid')['new_session'].cumsum()
    df_exp['unique_session_id'] = df_exp['visitorid'].astype(str) + '_' + df_exp['session_id'].astype(str)

    # Labeling
    session_labels = df_exp.groupby('unique_session_id')['event'].apply(lambda x: 1 if 'transaction' in x.values else 0)

    # Filtering
    df_filtered = df_exp[df_exp['event'] != 'transaction'].copy()
    sequences = df_filtered.groupby('unique_session_id')['event'].apply(list)
    sequences = sequences[sequences.apply(len) >= 2]
    labels = session_labels[sequences.index]

    # Balancing
    data = pd.DataFrame({'sequence': sequences, 'label': labels})
    data_majority = data[data.label == 0]
    data_minority = data[data.label == 1]
    
    if len(data_minority) == 0:
        return 0

    data_majority_downsampled = resample(data_majority, 
                                         replace=False,
                                         n_samples=len(data_minority),
                                         random_state=42) 
    data_balanced = pd.concat([data_majority_downsampled, data_minority])

    # Training
    observations = {'view': 0, 'addtocart': 1}
    
    def train_hmm(sequences, n_components=4):
        train_seq = []
        for seq in sequences:
            encoded_seq = [observations[obv] for obv in seq if obv in observations]
            if encoded_seq:
                train_seq.append(encoded_seq)
        
        if not train_seq:
            return None

        train_seq_fixed = np.concatenate(train_seq).reshape(-1, 1)
        lengths = [len(seq) for seq in train_seq]
        
        model = hmm.CategoricalHMM(n_components=n_components, n_iter=100, random_state=42)
        model.fit(train_seq_fixed, lengths)
        return model

    X_train, X_test, y_train, y_test = train_test_split(data_balanced['sequence'], data_balanced['label'], test_size=0.2, random_state=42)

    hmm_purchase = train_hmm(X_train[y_train == 1])
    hmm_no_purchase = train_hmm(X_train[y_train == 0])

    # Evaluation
    correct = 0
    total = 0

    for seq, true_label in zip(X_test, y_test):
        encoded_seq = [observations[obv] for obv in seq if obv in observations]
        if not encoded_seq:
            continue
            
        encoded_seq = np.array(encoded_seq).reshape(-1, 1)
        
        try:
            score_purchase = hmm_purchase.score(encoded_seq)
        except:
            score_purchase = -np.inf
            
        try:
            score_no_purchase = hmm_no_purchase.score(encoded_seq)
        except:
            score_no_purchase = -np.inf
            
        pred_label = 1 if score_purchase > score_no_purchase else 0
        
        if pred_label == true_label:
            correct += 1
        total += 1

    acc = correct/total
    print(f"Accuracy: {acc:.4f}")
    return acc

# Run Experiment
thresholds = [30, 60, 120]
results = {}

for t in thresholds:
    acc = run_experiment(t, df)
    results[t] = acc

print("\n--- Final Results ---")
for t, acc in results.items():
    print(f"Threshold {t} mins: Accuracy = {acc:.4f}")

best_threshold = max(results, key=results.get)
print(f"\nBest Threshold: {best_threshold} minutes")

--- Running Experiment with Threshold: 30 minutes ---
Accuracy: 0.9042
--- Running Experiment with Threshold: 60 minutes ---
Accuracy: 0.8972
--- Running Experiment with Threshold: 120 minutes ---
Accuracy: 0.8853

--- Final Results ---
Threshold 30 mins: Accuracy = 0.9042
Threshold 60 mins: Accuracy = 0.8972
Threshold 120 mins: Accuracy = 0.8853

Best Threshold: 30 minutes


## Final Sessionization
Based on the experiment above, we select the best threshold (30 minutes) for our final model.

In [5]:
# Calculate time difference between consecutive events for each visitor
df['time_diff'] = df.groupby('visitorid')['timestamp'].diff()

# Define session threshold (30 minutes)
session_threshold = pd.Timedelta(minutes=30)

# Create new session ID
df['new_session'] = (df['time_diff'].isna()) | (df['time_diff'] > session_threshold)
df['session_id'] = df.groupby('visitorid')['new_session'].cumsum()

# Create unique session identifier (visitorid + session_id)
df['unique_session_id'] = df['visitorid'].astype(str) + '_' + df['session_id'].astype(str)

print(f"Total unique sessions: {df['unique_session_id'].nunique()}")

Total unique sessions: 1761675


## Labeling and Filtering
We label each session as `1` (purchase) if it contains a 'transaction' event, and `0` otherwise. 
**Crucially**, we remove the 'transaction' events from the sequences used for training to prevent observation leakage.

In [6]:
# Label sessions: 1 if 'transaction' occurs in the session, 0 otherwise
session_labels = df.groupby('unique_session_id')['event'].apply(lambda x: 1 if 'transaction' in x.values else 0)

# Filter out 'transaction' events from the sequences themselves (Observation Leakage Fix)
df_filtered = df[df['event'] != 'transaction'].copy()

# Create sequences
sequences = df_filtered.groupby('unique_session_id')['event'].apply(list)

# Filter out short sequences (length < 2)
sequences = sequences[sequences.apply(len) >= 2]

# Align labels with filtered sequences
labels = session_labels[sequences.index]

print(f"Total sequences: {len(sequences)}")
print(f"Purchase sequences: {sum(labels)}")
print(f"No-purchase sequences: {len(labels) - sum(labels)}")

Total sequences: 381235
Purchase sequences: 12217
No-purchase sequences: 369018


## Data Balancing
Since purchase events are rare, we undersample the majority class (no-purchase) to create a balanced dataset.

In [7]:
# Combine sequences and labels into a DataFrame for resampling
data = pd.DataFrame({'sequence': sequences, 'label': labels})

# Separate majority and minority classes
data_majority = data[data.label == 0]
data_minority = data[data.label == 1]

# Undersample majority class
data_majority_downsampled = resample(data_majority, 
                                     replace=False,    # sample without replacement
                                     n_samples=len(data_minority), # match minority class size
                                     random_state=42) 

# Combine minority class with downsampled majority class
data_balanced = pd.concat([data_majority_downsampled, data_minority])

print("After balancing:")
print(data_balanced.label.value_counts())

After balancing:
label
0    12217
1    12217
Name: count, dtype: int64


## HMM Training
We train two separate HMMs: one for purchase sequences and one for no-purchase sequences.

In [8]:
# Map observations to integers
observations = {'view': 0, 'addtocart': 1} # 'transaction' is excluded

def train_hmm(sequences, n_components=4):
    train_seq = []
    for seq in sequences:
        encoded_seq = [observations[obv] for obv in seq if obv in observations]
        if encoded_seq: # Ensure not empty
            train_seq.append(encoded_seq)
    
    if not train_seq:
        return None

    # Flatten
    train_seq_fixed = np.concatenate(train_seq).reshape(-1, 1)
    # Lengths
    lengths = [len(seq) for seq in train_seq]
    
    model = hmm.CategoricalHMM(n_components=n_components, n_iter=100, random_state=42)
    model.fit(train_seq_fixed, lengths)
    return model

# Split data
X_train, X_test, y_train, y_test = train_test_split(data_balanced['sequence'], data_balanced['label'], test_size=0.2, random_state=42)

# Train HMM_purchase
print("Training HMM_purchase...")
hmm_purchase = train_hmm(X_train[y_train == 1])

# Train HMM_no_purchase
print("Training HMM_no_purchase...")
hmm_no_purchase = train_hmm(X_train[y_train == 0])

print("Training complete.")

Training HMM_purchase...
Training HMM_no_purchase...
Training complete.


## Evaluation
We classify test sequences by calculating the log-likelihood under both models and assigning the class with the higher likelihood.

In [9]:
print("Evaluating...")
correct = 0
total = 0

y_pred = []

for seq, true_label in zip(X_test, y_test):
    encoded_seq = [observations[obv] for obv in seq if obv in observations]
    if not encoded_seq:
        continue
        
    encoded_seq = np.array(encoded_seq).reshape(-1, 1)
    
    try:
        score_purchase = hmm_purchase.score(encoded_seq)
    except:
        score_purchase = -np.inf
        
    try:
        score_no_purchase = hmm_no_purchase.score(encoded_seq)
    except:
        score_no_purchase = -np.inf
        
    pred_label = 1 if score_purchase > score_no_purchase else 0
    y_pred.append(pred_label)
    
    if pred_label == true_label:
        correct += 1
        total += 1

print(f"Accuracy: {correct/total:.4f}")

print("Classification Report:")
print(classification_report(y_test, y_pred))

Evaluating...
Accuracy: 1.0000
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91      2460
           1       0.92      0.88      0.90      2427

    accuracy                           0.90      4887
   macro avg       0.91      0.90      0.90      4887
weighted avg       0.91      0.90      0.90      4887

